In [ ]:
SESSION_UUID = "REPLACE_WITH_SESSION_UUID"

import pathlib
WORKSPACES = pathlib.Path("/home/ubuntu/lab/workspaces")
SESSION_DIR = WORKSPACES / SESSION_UUID / f"session_{SESSION_UUID}"
assert SESSION_DIR.exists(), f"Workspace not found: {SESSION_DIR}"
print(f"Session: {SESSION_DIR}")

In [ ]:
import json
import pandas as pd

records = []
with open(SESSION_DIR / "frames.ndjson") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        r = json.loads(line)
        records.append({
            "frame_number"   : r["frame_number"],
            "frame_filename" : r["frame_filename"],
            "ts_app"         : r["ts_app"],
            "measured_fps"   : r.get("measured_fps"),
            "camera_obscured": r.get("camera_obscured", False),
            "thermal_state"  : r.get("thermal_state"),
            "battery_pct"    : r.get("battery_level_pct"),
            "has_depth"      : "depth_filename" in r,
        })

frames_df = pd.DataFrame(records).set_index("frame_number")
print(f"Loaded {len(frames_df)} frames")
print(frames_df.head())

In [ ]:
# ON-DEVICE NOTE: Per-frame processing is the expected on-device pattern.
# Operations applied here should not require future frame context.

from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def show_frame(frame_number):
    row = frames_df.loc[frame_number]
    img_path = SESSION_DIR / row["frame_filename"]
    img = Image.open(img_path)

    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    ax.imshow(img)
    ax.axis("off")

    info = (
        f"Frame {frame_number}  |  "
        f"FPS: {row['measured_fps']:.1f}  |  "
        f"Battery: {row['battery_pct']:.0f}%  |  "
        f"Thermal: {row['thermal_state']}  |  "
        f"{'\u26a0 OBSCURED' if row['camera_obscured'] else 'Clear'}"
    )
    ax.set_title(info, fontsize=10, color="red" if row["camera_obscured"] else "black")
    plt.tight_layout()
    plt.show()

# Show frame 0 to start
show_frame(0)

In [ ]:
from ipywidgets import interact, IntSlider

@interact(frame=IntSlider(min=0, max=len(frames_df)-1, step=1, value=0, continuous_update=False))
def scrub(frame):
    show_frame(frame)

In [ ]:
obscured = frames_df[frames_df["camera_obscured"] == True]
print(f"Obscured frames: {len(obscured)}")
if not obscured.empty:
    print(obscured[["frame_filename", "measured_fps", "thermal_state"]])

In [ ]:
depth_frames = frames_df[frames_df["has_depth"] == True]
if depth_frames.empty:
    print("No LiDAR depth data in this session")
else:
    print(f"LiDAR depth available for {len(depth_frames)} frames")
    # Show first depth frame alongside RGB
    first = depth_frames.index[0]
    row = frames_df.loc[first]
    rgb = Image.open(SESSION_DIR / row["frame_filename"])

    import json
    with open(SESSION_DIR / "frames.ndjson") as f:
        for line in f:
            r = json.loads(line.strip())
            if r["frame_number"] == first:
                depth_file = r["depth_filename"]
                break

    import numpy as np
    depth = np.array(Image.open(SESSION_DIR / depth_file))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.imshow(rgb); ax1.set_title(f"RGB \u2014 Frame {first}"); ax1.axis("off")
    im = ax2.imshow(depth, cmap="plasma"); ax2.set_title("Depth (mm)"); ax2.axis("off")
    plt.colorbar(im, ax=ax2, label="mm")
    plt.tight_layout(); plt.show()